In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from typing import List
import numpy as np
from tqdm import tqdm
from abc import abstractmethod
import plotly.express as px


In [ ]:
def styled_figure(data: List[go.Trace],
                  title: str = "Plot",
                  title_x: str = "",
                  title_y: str = "") -> go.Figure:
    """Create an already styled plotly figure
    Parameters
    ----------
    data : List[go.Trace]
        list of traces to appear in the figure
    title : str
        title of the figure (default: Plot)
    title_x : str
        title of the x-axis
    title_y : str
        title of the y-axis

    Returns
    -------
    go.Figure
        styled figure with all traces
    """
    layout = go.Layout(
        title=title,
        plot_bgcolor="#FFFFFF",
        hovermode="closest",
        width=800,
        height=600,
        hoverdistance=100,  # Distance to show hover label of data point
        spikedistance=1000,  # Distance to show spike
        showlegend=False,
        margin=dict(l=50, r=50, b=100, t=100, pad=2),
        xaxis=dict(
            title=title_x,
            linecolor="#BCCCDC",
            showspikes=False,  # Show spike line for X-axis
            # Format spike
            spikethickness=2,
            spikedash="dot",
            spikecolor="#999999",
            spikemode="marker",
            spikesnap="data",
            showline=True,
            linewidth=1,
            automargin=True,
        ),
        yaxis=dict(
            title=title_y,
            linecolor="#BCCCDC",
            showline=True,
            linewidth=1,
            automargin=True,
        ),
    )
    fig = go.Figure(data=data, layout=layout)
    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",
        # xaxis_title='input values',
        # yaxis_title='output values',
        legend=dict(yanchor="top", y=0.9, xanchor="right", x=0.95),
        title={
            "x": 0.5,
            "xanchor": "center"
        },
    )

    return fig


def line_scatter(
    visible: bool = True,
    x_lines: np.array = np.array([]),
    y_lines: np.array = np.array([]),
    name_line: str = "Predicted function",
) -> go.Scatter:
    """Create a line scatter very simple

    Parameters
    ----------
    visible : bool
        determine if line is visible (default: True)
    x_lines : np.ndarray
        dots on the x-axis, i.p. one dimensional array
    y_lines : np.ndarray
        dots on y-axis, i.p. one dimensional array
    name_line : str
        name of the line (default: Predicted function")

    Returns
    -------
    go.Scatter
        line scatter with coordinates specified in x/y-lines
    """
    return go.Scatter(
        visible=visible,
        line=dict(width=2),
        x=x_lines,
        y=y_lines,
        name=name_line,
    )


def dot_scatter(
    visible: bool = True,
    x_dots: np.array = np.array([]),
    y_dots: np.array = np.array([]),
    name_dots: str = "Observed points",
    fill="tonexty",
    text=None,
) -> go.Scatter:
    """Create a styled dot scatter very simple

    Parameters
    ----------
    visible : bool
        determine if line is visible (default: True)
    x_dots : np.ndarray
        dots on the x-axis, i.p. one dimensional array
    y_dots : np.ndarray
        dots on y-axis, i.p. one dimensional array
    name_dots : str
        name of the dots (default: Predicted function")
    fill : str
       fill property of Scatter (default: tonexty, i.e. fill to next trace)
    text : str
        TBA (default: None)

    Returns
    -------
    go.Scatter
        dot scatter with coordinates specified in x/y-lines
    """
    return go.Scatter(
        x=x_dots,
        visible=visible,
        y=y_dots,
        text=text,
        textposition="top center",
        mode="markers+text",
        name=name_dots,
        fillcolor="rgba(100, 100, 100, 0.2)",
        marker=dict(size=8),
    )

## Hyperparameters

In [ ]:
number_sample_points = 20**2
number_bootstrap = 1000
number_resampling = 100

## Iid Case

### Data generation

In [ ]:
# iid normal
variance = 10
mean = 4

sampled_points = []
means = []

# sample from distribution
for _ in tqdm(range(number_resampling)):
    sampled_points.append(
        np.random.multivariate_normal(
            mean=mean * np.ones(number_sample_points),
            cov=variance * np.identity(number_sample_points)))
    
print(f"True variance of emperical mean: {variance/number_sample_points}")

In [ ]:
means_resampled = [np.average(sampled_point) for sampled_point in sampled_points]
print(
    f"Resampled variance estimate: {np.var(means_resampled)}")
fig = px.histogram(means_resampled)
fig.show()

### Discrete bootstrap

In [ ]:
# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points[0])

for _ in tqdm(range(number_bootstrap)):
    resampled_points = [np.random.choice(sampled_points[0]-mean_bootstrapped) for _,_ in enumerate(sampled_points[0])] 
    means_bootstrapped.append(np.average(resampled_points))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

### Weighted bootstrap

In [ ]:
# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points[0])

for _ in tqdm(range(number_bootstrap)):
    random_weights = np.array([
        np.random.normal(loc=1, scale=1) for _, _ in enumerate(sampled_points[0])
    ])
    means_bootstrapped.append(
        np.average(random_weights * (sampled_points[0]-mean_bootstrapped)))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

## Stationary strong mixing sequence

In [ ]:
number_sample_points = 50**2
number_bootstrap = 5000
number_resampling = 1

### Data generation 

In [ ]:
mean = 1
variance = 2
a = 0.8

In [ ]:
# sample number_resampling-often sequences of length umber_sample_points from distribution
sampled_points_of_distributions = []

for _ in tqdm(range(number_resampling)):
    # sample number_sample_points-often from distribution (not iid): X_i = Y_i+a*Y_{i+1}
    sampled_points_of_distributions.append(
        np.random.multivariate_normal(
            mean=mean * np.ones(number_sample_points + 1),
            cov=variance * np.identity(number_sample_points + 1)))
    sampled_points_of_distributions[-1][:-2] += a * \
        sampled_points_of_distributions[-1][2:]
    sampled_points_of_distributions[-1] = sampled_points_of_distributions[
        -1][:-1]

# define original sample
sampled_points = sampled_points_of_distributions[0]

In [ ]:
# mean of X_0 (equal to mean of X_i since X_i stationary sequence)
mean = mean + a * mean

# calculate arithmetic mean minus actual mean in order to approximate distribution of G_n-G
means = []
means.append([
    np.average(sampled_points_of_distribution) - mean
    for sampled_points_of_distribution in sampled_points_of_distributions
])

print(np.std(means))

In [ ]:
# calculated variance
true_variance = (1+a)**2/number_sample_points*variance
print(f"True variance of emperical mean: {true_variance}")

### Discrete and weighted bootstrap for iid

In [ ]:
# Discrete
# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    resampled_points = [np.random.choice(sampled_points-mean_bootstrapped) for _,_ in enumerate(sampled_points)] 
    means_bootstrapped.append(np.average(resampled_points))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

In [ ]:
# Weighted
# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = np.array([
        np.random.normal(loc=1, scale=1) for _, _ in enumerate(sampled_points)
    ])
    means_bootstrapped.append(
        np.average(random_weights * (sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

### blockwise bootstrap discrete 

In [ ]:
# block bootstrap
block_length = int(number_sample_points**(1 / 5))  # =l_n
number_blocks = int(number_sample_points / block_length)  # =k
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    weights = [
        np.random.uniform(low=0, high=number_sample_points - block_length)
        for _ in range(number_blocks)
    ]

    # calculate mean
    means_bootstrapped.append(1 / number_sample_points * np.sum([
        np.sum(sampled_points[int(weight):int(weight) + block_length - 1])
        for weight in weights
    ]) - mean_bootstrapped)

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

### Recursive sequence

In [ ]:
alpha = 2**(1/2)-1


def rho(i):
    return 1-i**-alpha


def V():
    normal_weights = [
        np.random.normal(loc=0, scale=1)
        for i in range(number_sample_points)
    ]

    random_weights = [
        normal_weights[0]+1
    ]
    for i, normal_weight in enumerate(normal_weights[1:]):
        random_weights += [1+rho(i+1)*(random_weights[i]-1) +
                                 (1-rho(i+1)**2)**(1/2)*normal_weight]
    return random_weights

# bootstrap


# bootstrapped samples
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)
for _ in tqdm(range(number_bootstrap)):
    random_weights = V()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                                      np.sum(random_weights * sampled_points) -
                                      mean_bootstrapped)
print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

#### Recursive sequence benchmark different alpha

In [ ]:
#alpha = 0.7
#alpha = 2**(1/2)-1
def rho(i):
    return 1-i**-alpha

def V():
    normal_weights = [
        np.random.normal(loc=0, scale=1)
        for i in range(number_sample_points)
    ]
    
    
    random_weights = [
        normal_weights[0]+1
    ]
    for i,normal_weight in enumerate(normal_weights[1:]):
        random_weights+=[1+rho(i+1)*(random_weights[i]-1)+(1-rho(i+1)**2)**(1/2)*normal_weight]
    return random_weights

# bootstrap

# bootstrapped samples
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

list_calculated_variances_with_different_alpha = {}
#for alpha in [2**(1/2)-1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]:
for alpha in [2**(1/2)-1,0.9]:
    list_calculated_variances = []
    for _ in tqdm(range(1)):
    #for _ in tqdm(range(100)):
        for _ in range(number_bootstrap):
            random_weights = V()
            means_bootstrapped.append(1 / (np.sum(random_weights)) *
                                      np.sum(random_weights * sampled_points) -
                                      mean_bootstrapped)

        list_calculated_variances.append(np.var(means_bootstrapped))
    
    list_calculated_variances_with_different_alpha[alpha] = list_calculated_variances
    

In [ ]:
list_calculated_variances = {alpha:[np.mean(list_calculated_variances_with_different_alpha[alpha])
                                    ,np.std(list_calculated_variances_with_different_alpha[alpha])]
                            for alpha in list_calculated_variances_with_different_alpha.keys()}

In [ ]:
list_calculated_variances

In [ ]:
px.bar(title="Mean",x=list_calculated_variances.keys(),y=np.array(list(list_calculated_variances.values())).T[0].T-true_variance)

In [ ]:
px.bar(title="Variance",x=list_calculated_variances.keys(),y=np.array(list(list_calculated_variances.values())).T[1].T)

### blockwise bootstrap moving average

In [ ]:
q = 2 / (3 * block_length) + 1 / (3 * block_length**3)


def triangle_window(m, j):
    return 1 / m * (1 - np.abs(j) / m)


def V():
    gamma_weights = [
        np.random.gamma(q, 1/q)
        for _ in range(number_sample_points + 2 * block_length) # !!
    ]

    random_weights = np.array([
        np.sum([
            triangle_window(block_length, j) * gamma_weights[t - j]
            for j in range(-block_length, block_length + 1)
        ]) for t in range(number_sample_points)
    ])
    return random_weights

In [ ]:
# bootstrap

# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = V()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                              np.sum(random_weights * sampled_points) -
                              mean_bootstrapped)
    # means_bootstrapped.append(np.average(random_weights*(sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

### blockwise bootstrap indicator

In [ ]:
q1 = 1/(2*block_length)


def indicator(m, j):
    return 1/(2*m)


def V1():
    gamma_weights = [
        np.random.gamma(q1, 1/q1)
        for _ in range(number_sample_points + 2 * block_length) 
    ]

    random_weights = np.array([
        np.sum([
            indicator(block_length, j) * gamma_weights[t - j]
            for j in range(-block_length, block_length + 1)
        ]) for t in range(number_sample_points)
    ])
    return random_weights

In [ ]:
# bootstrap

# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = V1()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                              np.sum(random_weights * sampled_points) -
                              mean_bootstrapped)
    # means_bootstrapped.append(np.average(random_weights*(sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))
fig = px.histogram(means_bootstrapped)
fig.show()

---

### Online bootstrap

In [ ]:
def calculate_block_length(n):
    return int(n**(1/5))

In [ ]:
def b(j):
    return (2 * calculate_block_length(j))**(-1 / 2)


def V2():
    normal_weights = [
        np.random.normal(loc=0, scale=1)
        for _ in range(calculate_block_length(number_sample_points) + number_sample_points)
    ]
    random_weights = [
        1 + b(j) * np.sum(normal_weights[j - calculate_block_length(j):j +
                                         calculate_block_length(j)])
                          for j in range(1, number_sample_points + 1)
    ]
    return random_weights

In [ ]:
# bootstrap

# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = V2()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                              np.sum(random_weights * sampled_points) -
                              mean_bootstrapped)
    # means_bootstrapped.append(np.average(random_weights*(sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))

In [ ]:
def b(j):
    return (2 * calculate_block_length(j))**(-1 / 2)


def V2():
    normal_weights = [
        np.random.normal(loc=0, scale=1)
        for _ in range(calculate_block_length(number_sample_points) + number_sample_points)
    ]
    random_weights = [
        1 + b(j) * np.sum(normal_weights[:j +
                                         calculate_block_length(j)])
                          for j in range(1, number_sample_points + 1)
    ]
    return random_weights

In [ ]:
# ERROR!

# bootstrap

# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = V2()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                              np.sum(random_weights * sampled_points) -
                              mean_bootstrapped)
    # means_bootstrapped.append(np.average(random_weights*(sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))

## Recursive sequence

## Playground

In [ ]:
from scipy.stats import norm,expon

In [ ]:
def V3():
    normal_weights = [
        np.random.normal(loc=0, scale=1)
        for _ in range(calculate_block_length(number_sample_points) + number_sample_points)
    ]
    random_weights = np.array([
        1 + b(j) * np.sum(normal_weights[j - calculate_block_length(j):j +
                                         calculate_block_length(j)])
                          for j in range(1, number_sample_points + 1)
    ])
    return expon.ppf(norm.cdf(random_weights-1))

In [ ]:
# bootstrap

# bootstrapped samples
sampled_points_of_distributions_bootstrapped = []
means_bootstrapped = []

# calculate mean
mean_bootstrapped = np.average(sampled_points)

for _ in tqdm(range(number_bootstrap)):
    random_weights = V3()
    means_bootstrapped.append(1 / (np.sum(random_weights)) *
                              np.sum(random_weights * sampled_points) -
                              mean_bootstrapped)
    # means_bootstrapped.append(np.average(random_weights*(sampled_points-mean_bootstrapped)))

print(np.var(means_bootstrapped))